In [2]:
!pip install -q langchain langchain-groq langchain-core pydantic python-dotenv

In [4]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

Enter your Groq API key: ··········


In [5]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

In [29]:
class LogAnalysis(BaseModel):

    summary: str = Field(
        description="Overall summary of the application log"
    )

    errors: list[str] = Field(
        description="Important errors found in the logs"
    )

    warnings: list[str] = Field(
        description="Important warnings found in the logs"
    )

    repeated_errors: list[str] = Field(
        description="Errors that occurred multiple times"
    )

    possible_causes: list[str] = Field(
        description="Possible causes of the detected problems"
    )

    suggestions: list[str] = Field(
        description="Practical troubleshooting suggestions"
    )

In [30]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

In [31]:
structured_llm = llm.with_structured_output(LogAnalysis)

In [32]:
prompt = ChatPromptTemplate.from_template("""
You are an AI application log analyzer.

Analyze the provided application logs carefully.

Your tasks are:

1. Identify important errors.
2. Identify important warnings.
3. Identify errors that occur repeatedly.
4. Suggest possible causes for the detected problems.
5. Suggest practical troubleshooting steps.
6. Provide a concise overall summary.

The application has already calculated these statistics:

Total log entries: {total_entries}
Errors: {error_count}
Warnings: {warning_count}
Info messages: {info_count}

Important rules:

- Base your analysis only on the provided logs.
- Do not invent specific facts that are not present in the logs.
- Do not change the provided statistics.
- If there are no errors, return an empty list for errors.
- If there are no warnings, return an empty list for warnings.
- Focus on explaining the problems found in the logs.
- Keep troubleshooting suggestions practical and relevant.

Application Logs:

{logs}
""")

In [33]:
chain = prompt | structured_llm

In [34]:
log_content = """
2026-09-29 10:20:01 INFO Server started successfully
2026-09-29 10:20:03 INFO User authentication service started
2026-09-29 10:20:05 WARNING Database response time is high
2026-09-29 10:20:08 ERROR Connection refused to database
2026-09-29 10:20:09 ERROR Connection refused to database
2026-09-29 10:20:12 ERROR Database connection timeout
2026-09-29 10:20:15 INFO Retrying database connection
2026-09-29 10:20:18 ERROR Connection refused to database
2026-09-29 10:20:20 INFO Database connection restored
2026-09-29 10:20:25 WARNING Memory usage is above 80 percent
2026-09-29 10:20:30 INFO Server running normally
"""

with open("application.log", "w") as file:
    file.write(log_content)

print("application.log created successfully!")

application.log created successfully!


In [36]:
with open("application.log", "r", encoding="utf-8") as file:
    log_text = file.read()

print(log_text)


2026-09-29 10:20:01 INFO Server started successfully
2026-09-29 10:20:03 INFO User authentication service started
2026-09-29 10:20:05 WARNING Database response time is high
2026-09-29 10:20:08 ERROR Connection refused to database
2026-09-29 10:20:09 ERROR Connection refused to database
2026-09-29 10:20:12 ERROR Database connection timeout
2026-09-29 10:20:15 INFO Retrying database connection
2026-09-29 10:20:18 ERROR Connection refused to database
2026-09-29 10:20:20 INFO Database connection restored
2026-09-29 10:20:25 WARNING Memory usage is above 80 percent
2026-09-29 10:20:30 INFO Server running normally



In [37]:
log_lines = [
    line.strip()
    for line in log_text.splitlines()
    if line.strip()
]

error_count = 0
warning_count = 0
info_count = 0

for line in log_lines:

    if " ERROR " in line:
        error_count += 1

    elif " WARNING " in line:
        warning_count += 1

    elif " INFO " in line:
        info_count += 1


log_stats = {
    "total_entries": len(log_lines),
    "errors": error_count,
    "warnings": warning_count,
    "info": info_count
}

print("Total entries:", log_stats["total_entries"])
print("Errors:", log_stats["errors"])
print("Warnings:", log_stats["warnings"])
print("Info:", log_stats["info"])

Total entries: 11
Errors: 4
Warnings: 2
Info: 5


In [38]:
result = chain.invoke({
    "logs": log_text,
    "total_entries": log_stats["total_entries"],
    "error_count": log_stats["errors"],
    "warning_count": log_stats["warnings"],
    "info_count": log_stats["info"]
})

In [39]:
print("========== LOG SUMMARY ==========")
print(result.summary)

print("\n========== ERRORS ==========")
for error in result.errors:
    print("🔴", error)

print("\n========== WARNINGS ==========")
for warning in result.warnings:
    print("⚠️", warning)

print("\n========== REPEATED ERRORS ==========")
for error in result.repeated_errors:
    print("🔁", error)

print("\n========== POSSIBLE CAUSES ==========")
for cause in result.possible_causes:
    print("🔍", cause)

print("\n========== SUGGESTIONS ==========")
for suggestion in result.suggestions:
    print("🛠️", suggestion)

========== LOG SUMMARY ==========
The server and authentication services started successfully, but the application encountered repeated "Connection refused to database" errors and a timeout, indicating connectivity issues with the database. Warnings about high database response time and elevated memory usage were logged. After retrying, the database connection was restored and the server ran normally by 10:20:30.

========== ERRORS ==========
🔴 Connection refused to database
🔴 Database connection timeout

========== WARNINGS ==========
⚠️ Database response time is high
⚠️ Memory usage is above 80 percent

========== REPEATED ERRORS ==========
🔁 Connection refused to database

========== POSSIBLE CAUSES ==========
🔍 Database service may be down or unreachable due to network/firewall issues
🔍 Incorrect database host/port or credentials
🔍 High load causing timeouts
🔍 Resource contention or insufficient resources

========== SUGGESTIONS ==========
🛠️ Verify that the database server is runn